# Задача 7.1. Остывание стержня: что гарантирует устойчивость
**Зелёный и жёлтый уровни**

Рассматриваем безразмерное уравнение теплопроводности
$$u_t=u_{xx},\qquad 0<x<1,\qquad u(0,t)=u(1,t)=0.$$
Сначала проверим явную схему FTCS и неявную схему Эйлера на гладком профиле, для которого есть точное решение. Затем на фиксированной пространственной сетке добавим схему Кранка—Николсона и посмотрим, как крупный шаг по времени влияет на остывание участка с резкими краями.

Для запуска нужны NumPy и Matplotlib. Ячейки выполняются сверху вниз. Графики и численные результаты сохранены.

Материалы: [условие задания 7.1](https://github.com/yartmihant/Students6Year2026/blob/86251f18f04ea2ce16506d96d6735db483c78e02/README.md) и [конспект занятия 7](https://github.com/yartmihant/Students6Year2026/blob/86251f18f04ea2ce16506d96d6735db483c78e02/lesson7/lesson_07_theory.md). Условия расчётов взяты из обновлённого README.

## 1. Пространственная сетка и шаг по времени

Пусть $N$ — число интервалов, $h=1/N$, $x_j=jh$, $j=0,\dots,N$. Граничные узлы на каждом слое равны нулю. Во внутренних узлах используем
$$L_hU_j=\frac{U_{j-1}-2U_j+U_{j+1}}{h^2}.$$
Все три схемы удобно записать через один параметр $\theta$:
$$
U^{n+1}-U^n=\Delta t\big[(1-\theta)L_hU^n+\theta L_hU^{n+1}\big].
$$

| Схема | $\theta$ | Как берётся пространственный оператор |
|---|---:|---|
| FTCS | 0 | На известном слое |
| Неявный Эйлер | 1 | На новом слое |
| Кранк—Николсон | $1/2$ | Среднее старого и нового слоёв |

Для $\theta>0$ решаем трёхдиагональную систему методом прогонки. При $r=\Delta t/h^2$ диагональ равна $1+2\theta r$, соседние диагонали равны $-\theta r$. Если $r$ не меняется, коэффициенты исключения можно посчитать один раз и использовать на всех шагах.

Запрошенный шаг равен $rh^2$. Если конечный момент не кратен шагу, последний шаг сокращается и для него заново вычисляется $r$. Для FTCS используем только $r=0.4$, так как условие устойчивости на этой задаче — $r\le1/2$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

theta = {"FTCS": 0.0, "BE": 1.0, "CN": 0.5}
labels = {"FTCS": "FTCS", "BE": "Неявный Эйлер", "CN": "Кранк—Николсон"}
colors = {"FTCS": "#64864e", "BE": "#2f7791", "CN": "#b25165"}
plt.rcParams.update({
    "figure.dpi": 115, "font.size": 10, "axes.grid": True, "grid.alpha": 0.18,
    "axes.spines.top": False, "axes.spines.right": False,
})


def prepare_tridiagonal(m, diag, side):
    pivots = np.empty(m)
    factors = np.empty(m-1)
    pivots[0] = diag
    for j in range(1, m):
        factors[j-1] = side/pivots[j-1]
        pivots[j] = diag-factors[j-1]*side
    return pivots, factors, side


def solve_tridiagonal(rhs, prepared):
    pivots, factors, side = prepared
    z = rhs.copy()
    for j in range(1, len(z)):
        z[j] -= factors[j-1]*z[j-1]
    z[-1] /= pivots[-1]
    for j in range(len(z)-2, -1, -1):
        z[j] = (z[j]-side*z[j+1])/pivots[j]
    return z


def evolve(initial, method, ratio, t_stop):
    n_cells = len(initial)-1
    h = 1.0/n_cells
    nominal_dt = ratio*h*h
    full = int(np.floor(t_stop/nominal_dt + 1e-11))
    t = nominal_dt*np.arange(full+1)
    steps = [nominal_dt]*full
    if t_stop-t[-1] > 1e-14:
        steps.append(t_stop-t[-1])
        t = np.r_[t, t_stop]
    t[-1] = t_stop

    values = np.zeros((len(t), n_cells+1))
    values[0, 1:-1] = initial[1:-1]
    weight = theta[method]
    matrices = {}
    for k, dt in enumerate(steps):
        r_now = dt/(h*h)
        if weight == 0 and r_now > 0.5:
            raise ValueError("Шаг FTCS выходит за условие устойчивости.")
        old = values[k]
        rhs = old[1:-1] + (1-weight)*r_now*(old[:-2]-2*old[1:-1]+old[2:])
        if weight:
            if r_now not in matrices:
                matrices[r_now] = prepare_tridiagonal(
                    n_cells-1, 1+2*weight*r_now, -weight*r_now
                )
            rhs = solve_tridiagonal(rhs, matrices[r_now])
        values[k+1, 1:-1] = rhs
    return dict(t=t, u=values, dt=nominal_dt, r=ratio)


def at_time(run, moment):
    j = int(np.argmin(np.abs(run["t"]-moment)))
    if abs(run["t"][j]-moment) > 1e-12:
        raise ValueError("В расчёте нет слоя в заданный момент.")
    return run["u"][j]

## 2. Зелёный уровень: гладкий профиль

Начальные данные состоят из двух синусоид. Для гармоники $\sin(m\pi x)$ уравнение даёт затухание амплитуды как $\exp[-(m\pi)^2t]$. Поэтому точный профиль:
$$u(x,t)=e^{-\pi^2t}\sin(\pi x)+0.3e^{-25\pi^2t}\sin(5\pi x).$$
Пятая гармоника меняется быстрее в пространстве и имеет большую кривизну при той же амплитуде. Её показатель затухания в 25 раз больше, чем у первой. Это физическое сглаживание: оно есть уже в точном решении.

Для $N=25,50,100$ и обоих методов берём $\Delta t=0.4h^2$. Для каждого момента $t=0.001,\ 0.01,\ 0.1$ начинаем с одних и тех же начальных данных и, если нужно, сокращаем последний шаг. Считаем максимальную ошибку по узлам $E_\infty$ и порядок $p=\log_2(E_N/E_{2N})$. При такой связи шагов первый порядок по времени вместе со вторым порядком по пространству даёт ожидаемое поведение $O(h^2)$.

In [ ]:
def smooth_exact(x, t):
    return (np.exp(-np.pi**2*t)*np.sin(np.pi*x)
            + 0.3*np.exp(-25*np.pi**2*t)*np.sin(5*np.pi*x))


green_times = [0.001, 0.01, 0.1]
green_counts = [25, 50, 100]
green_results, green_errors = {}, {}
print(f"{'Схема':>16} {'t':>8} {'E(25)':>13} {'E(50)':>13} {'E(100)':>13} "
      f"{'p 25→50':>10} {'p 50→100':>10}")
for method in ["FTCS", "BE"]:
    for t in green_times:
        errors = []
        for N in green_counts:
            x = np.linspace(0, 1, N+1)
            run = evolve(smooth_exact(x, 0), method, 0.4, t)
            y = run["u"][-1]
            green_results[method, t, N] = (x, y)
            errors.append(np.max(np.abs(y-smooth_exact(x, t))))
        green_errors[method, t] = np.array(errors)
        p = np.log2(np.array(errors[:-1])/errors[1:])
        print(f"{labels[method]:>16} {t:8.3f} {errors[0]:13.6e} {errors[1]:13.6e} "
              f"{errors[2]:13.6e} {p[0]:10.3f} {p[1]:10.3f}")

x_dense = np.linspace(0, 1, 501)
fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.8), sharey=True,
                         constrained_layout=True)
for ax, t in zip(axes, green_times):
    ax.plot(x_dense, smooth_exact(x_dense, t), "k-", lw=1.4, label="Точное решение")
    for method, marker, stride in [("FTCS", "o", 2), ("BE", "x", 1)]:
        x, y = green_results[method, t, 25]
        ax.plot(x[::stride], y[::stride], linestyle="none", marker=marker,
                ms=4.3, color=colors[method], label=labels[method])
    ax.set(xlabel="x", xlim=(0, 1), title=f"t = {t:g}")
axes[0].set_ylabel("Температура")
axes[0].legend(fontsize=8, loc="lower center")
fig.suptitle("Гладкое остывание: точная кривая и узлы грубой сетки N = 25")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(9.8, 3.9), constrained_layout=True)
hs = 1/np.array(green_counts)
for ax, method in zip(axes, ["FTCS", "BE"]):
    for t, color in zip(green_times, ["#9b6b34", "#75499b", "#2f7791"]):
        ax.loglog(hs, green_errors[method, t], "o-", color=color, label=f"t = {t:g}")
    ax.loglog(hs, green_errors[method, 0.1][-1]*(hs/hs[-1])**2, "k:",
              label=r"Ориентир $h^2$")
    ax.set(xlabel="Размер интервала h", ylabel=r"$E_\infty$", title=labels[method])
    ax.legend(fontsize=8)
plt.show()

На рисунке показаны численные узлы самой грубой сетки, чтобы различия с точной кривой было проще увидеть. Таблица и график ошибок используют все три сетки. При $t=0.1$ оценки порядка близки к двум. При $t=0.001$ на сетке $N=25$ всего два шага, причём последний короче номинального; на этой грубой сетке оценки ещё далеки от асимптотических. При сгущении и они приближаются к двум.

## 3. Жёлтый уровень: нагретая полоса и малый шаг для сравнения

Теперь $U(x,0)=1$ на $[0.4,0.6]$ и 0 вне него. Фиксируем $N=100$, $h=0.01$. Включаем оба края, то есть 21 узел с номерами 40–60. В трёх опытах $r=0.4,2,10$, чему соответствуют $\Delta t=4\cdot10^{-5},2\cdot10^{-4},10^{-3}$. При $r=2,10$ используем только неявные схемы; FTCS добавляем при $r=0.4$.

Опорный расчёт — Кранк—Николсон при $r=0.02$ ($\Delta t=2\cdot10^{-6}$) на **той же пространственной сетке**. Так мы проверяем влияние шага по времени. Чтобы проверить точность опорного расчёта, независимо решим систему $\dot U=L_hU$ в дискретном синус-базисе. Для номера гармоники $m$ её собственное значение равно
$$\lambda_m=-\frac{4}{h^2}\sin^2\left(\frac{m\pi}{2N}\right).$$
Коэффициент этой гармоники умножается на $e^{\lambda_m t}$. Это точное решение **системы на сетке**, а не непрерывного уравнения. Сеточные значения нигде не обрезаются.

In [ ]:
N_strip, final_time = 100, 0.012
x_strip = np.linspace(0, 1, N_strip+1)
u_strip = ((np.arange(N_strip+1) >= 40) & (np.arange(N_strip+1) <= 60)).astype(float)
ratios = [0.4, 2.0, 10.0]
ratio_colors = {0.4: "#64864e", 2.0: "#9b6b34", 10.0: "#75499b"}
yellow_runs = {(method, r): evolve(u_strip, method, r, final_time)
               for r in ratios for method in ["BE", "CN"]}
yellow_runs["FTCS", 0.4] = evolve(u_strip, "FTCS", 0.4, final_time)
reference = evolve(u_strip, "CN", 0.02, final_time)

modes = np.arange(1, N_strip)
sines = np.sin(np.pi*np.outer(modes, modes)/N_strip)
initial_modes = (2.0/N_strip)*(sines.T @ u_strip[1:-1])
eigenvalues = -4*N_strip**2*np.sin(np.pi*modes/(2*N_strip))**2


def exact_on_grid(t):
    return np.r_[0, sines @ (initial_modes*np.exp(eigenvalues*t)), 0]


common_times = [0.001, 0.002, 0.004, 0.012]
ref_errors = np.array([np.max(np.abs(at_time(reference, t)-exact_on_grid(t)))
                       for t in common_times])
print("Проверка малого шага по точному решению системы на сетке:")
for t, error in zip(common_times, ref_errors):
    print(f"t = {t:7.4f}: E_inf = {error:.3e}")

## 4. Первые слои и локальные колебания

В каждой строке первого рисунка сравниваются решения **в один и тот же момент** $t=\Delta t$ для указанного $r$. Моменты между строками различаются: это сравнение поведения после одного шага. Отдельно ниже сопоставим разные $r$ при фиксированном физическом времени.

Чтобы выявить колебания около левого края, помимо минимума и максимума, измеряем
$$D(t_n)=U_{40}^n-U_{39}^n.$$
Изначально $D=1$. Положительный перепад означает, что узел с внутренней стороны полосы теплее. При $D<0$ порядок температур на этой паре обращается. Это может происходить даже тогда, когда все температуры лежат между нулём и единицей.

In [ ]:
def draw_layer(ax, r, n, window, add_labels=False):
    run = yellow_runs["CN", r]
    t = run["t"][n]
    ax.plot(x_strip, at_time(reference, t), "k--", lw=1.5, label="Малый шаг")
    for method in ["BE", "CN", "FTCS"]:
        if (method, r) in yellow_runs:
            u = yellow_runs[method, r]["u"][n]
            ax.plot(x_strip, u, color=colors[method], lw=1.5, marker=".",
                    ms=3.3, label=labels[method])
    ax.set(xlim=window, ylim=(-0.05, 1.05), xlabel="x")
    if add_labels:
        ax.legend(fontsize=7.5, loc="upper right")


fig, axes = plt.subplots(3, 3, figsize=(12, 8), sharey=True,
                         constrained_layout=True)
windows = [(0, 1), (0.34, 0.46), (0.54, 0.66)]
for i, r in enumerate(ratios):
    t = yellow_runs["CN", r]["t"][1]
    for j, (window, area) in enumerate(zip(windows, ["Весь стержень", "Левый край", "Правый край"])):
        ax = axes[i, j]
        draw_layer(ax, r, 1, window, add_labels=(i == j == 0))
        ax.set_title(area if j else f"r = {r:g}; t = {t:.5f}")
    axes[i, 0].set_ylabel("Температура")
fig.suptitle("Первый слой: весь профиль и два увеличенных края")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(11.6, 3.8), sharey=True,
                         constrained_layout=True)
for n, ax in enumerate(axes, 1):
    draw_layer(ax, 10.0, n, (0.33, 0.47), add_labels=(n == 1))
    cn = yellow_runs["CN", 10.0]["u"][n]
    d = cn[40]-cn[39]
    ax.axvline(0.4, color="0.7", ls=":", lw=1)
    ax.set_title(f"Слой {n}, t = {n*0.001:.3f}; D(CN) = {d:+.3f}")
axes[0].set_ylabel("Температура")
fig.suptitle("Три первых слоя при r = 10: перепад у левого края меняет знак")
plt.show()

print(f"{'Схема':>16} {'r':>6} {'min после 1 шага':>18} {'max после 1 шага':>18} "
      f"{'D₁':>12} {'D₂':>12}")
for (method, r), run in yellow_runs.items():
    u = run["u"]
    d = u[:, 40]-u[:, 39]
    print(f"{labels[method]:>16} {r:6g} {u[1].min():18.9f} {u[1].max():18.9f} "
          f"{d[1]:+12.6f} {d[2]:+12.6f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.0, 4.0), constrained_layout=True)
ref_u, ref_t = reference["u"], reference["t"]
for ax, y in zip(axes, [ref_u.min(axis=1), ref_u.max(axis=1), ref_u[:,40]-ref_u[:,39]]):
    ax.plot(ref_t, y, "k:", lw=1.6, label="Малый шаг")
for (method, r), run in yellow_runs.items():
    u = run["u"]
    style = {"BE": "--", "CN": "-", "FTCS": "-."}[method]
    color = ratio_colors[r] if method != "FTCS" else "#444444"
    for ax, y in zip(axes, [u.min(axis=1), u.max(axis=1), u[:,40]-u[:,39]]):
        ax.plot(run["t"], y, style, color=color, lw=1.25,
                label=f"{labels[method]}, r={r:g}")
axes[0].set(title="Минимум, включая концы", ylabel=r"$\min_j U_j$",
            ylim=(-0.03, 0.03), xlim=(0, final_time))
axes[1].set(title="Максимум температуры", ylabel=r"$\max_j U_j$",
            xlim=(0, final_time))
axes[2].axhline(0, color="0.5", ls=":", lw=0.8)
axes[2].set(title="Перепад у левого края", ylabel=r"$U_{40}-U_{39}$", xlim=(0, 0.006))
axes[1].legend(fontsize=6.8, loc="lower left")
for ax in axes:
    ax.set_xlabel("t")
plt.show()

В этом опыте глобальный минимум равен нулю из-за холодных концов, а максимум не превышает начальной единицы. Но на первом слое Кранка—Николсона при $r=2$ перепад у левого края отрицателен. При $r=10$ он меняется примерно от $-0.564$ на первом слое до $+0.584$ на втором. Это искусственное колебание профиля, хотя отрицательной температуры в данном опыте нет. На правом краю наблюдается зеркальная картина.

## 5. Разные шаги в одинаковые моменты времени

Для каждого $r$ сравниваем обе неявные схемы с опорным расчётом при одинаковом $t$. Ниже приведены максимальные отличия от малого шага:
$$E_{\Delta t}(t)=\max_j|U_j(t)-U_j^{\rm ref}(t)|.$$
На графике отмечен левый край, где различия особенно заметны. Численные линии соединяют узлы одной и той же сетки.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11.8, 6.1), sharex=True, sharey=True,
                         constrained_layout=True)
for i, t in enumerate([0.001, 0.004]):
    for j, r in enumerate(ratios):
        ax = axes[i, j]
        ax.plot(x_strip, at_time(reference, t), "k--", lw=1.55, label="Малый шаг")
        for method in ["BE", "CN"]:
            ax.plot(x_strip, at_time(yellow_runs[method, r], t), ".-",
                    ms=3.0, lw=1.4, color=colors[method], label=labels[method])
        ax.set(xlim=(0.27, 0.53), ylim=(-0.05, 1.05), xlabel="x", title=f"t = {t:g}, r = {r:g}")
    axes[i, 0].set_ylabel("Температура")
axes[0, 0].legend(fontsize=7.8, loc="upper left")
fig.suptitle("Один и тот же физический момент для всех трёх шагов в строке")
plt.show()

print(f"{'Схема':>16} {'r':>6}" + "".join(f"{t:>13g}" for t in common_times))
time_errors = {}
for (method, r), run in yellow_runs.items():
    errors = np.array([np.max(np.abs(at_time(run, t)-at_time(reference, t)))
                       for t in common_times])
    time_errors[method, r] = errors
    print(f"{labels[method]:>16} {r:6g}" + "".join(f"{e:13.5e}" for e in errors))

При $t=0.001$ и $r=10$ отличие Кранка—Николсона от малого шага примерно $0.327$, неявного Эйлера — $0.0621$. Неявный Эйлер здесь даёт гладкий профиль, но крупный шаг тоже меняет его заметно. При $r=0.4$ Кранк—Николсон в выбранные моменты гораздо ближе к опорному расчёту, чем неявный Эйлер. Значит, выигрыш от второго порядка по времени проявляется при достаточно малом шаге.

## 6. Почему устойчивость допускает колебания

Пусть $\mu=-\lambda_m\Delta t\ge0$ — величина, определяющая изменение отдельной дискретной гармоники за шаг. Её точный множитель $e^{-\mu}$. Численные множители:
$$G_{\rm BE}=\frac{1}{1+\mu},\qquad
G_{\rm CN}=\frac{1-\mu/2}{1+\mu/2}.$$
У обеих схем $|G|\le1$. Поскольку синусоидальные моды ортогональны, это ограничивает рост нормы решения. Однако для Кранка—Николсона при $\mu>2$ множитель отрицательный, а при больших $\mu$ близок к $-1$. Короткая волна тогда меняет знак на каждом шаге и затухает слишком медленно. При разрывных начальных данных таких волн много. Неявный Эйлер, у которого $G$ положителен и стремится к нулю, подавляет их гораздо сильнее.

На рисунке ниже сравним множители и проследим одну из быстрых мод в рассчитанном профиле. Знак моды меняется, но сама температура — сумма всех мод и не обязана становиться отрицательной.

In [ ]:
mu = np.linspace(0, 42, 501)
fast = -eigenvalues[-1]*yellow_runs["CN", 10.0]["dt"]
g_cn = (1-fast/2)/(1+fast/2)
g_be = 1/(1+fast)
print(f"Мода m={N_strip-1}, r=10: μ={fast:.4f}, "
      f"точно={np.exp(-fast):.3e}, BE={g_be:.5f}, CN={g_cn:.5f}")
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4), constrained_layout=True)
axes[0].plot(mu, np.exp(-mu), "k--", label="Точное затухание")
axes[0].plot(mu, 1/(1+mu), color=colors["BE"], label="Неявный Эйлер")
axes[0].plot(mu, (1-mu/2)/(1+mu/2), color=colors["CN"], label="Кранк—Николсон")
axes[0].axhline(0, color="0.6", lw=0.8)
axes[0].set(xlabel=r"$\mu=-\lambda_m\Delta t$", ylabel="Множитель за шаг",
            title="Быстрые гармоники и большой шаг")
axes[0].legend(fontsize=8)
n_plot = np.arange(7)
for method in ["BE", "CN"]:
    data = yellow_runs[method, 10.0]["u"][:len(n_plot), 1:-1]
    mode_amplitude = (2.0/N_strip)*(data @ sines[:, -1])/initial_modes[-1]
    axes[1].plot(n_plot, mode_amplitude, "o-", color=colors[method], label=labels[method])
axes[1].axhline(0, color="0.6", lw=0.8)
axes[1].set(xlabel="Номер слоя", ylabel="Амплитуда / начальная амплитуда",
            title=f"Из профиля: мода {N_strip-1}, r = 10", xticks=n_plot)
axes[1].legend(fontsize=8)
plt.show()

## Выводы

1. Гладкий профиль физически сглаживается, потому что пятая гармоника затухает в 25 раз быстрее первой. При $\Delta t=0.4h^2$ ошибки FTCS и неявного Эйлера уменьшаются примерно как $h^2$ при сгущении сетки.
2. При резком начальном профиле большой шаг Кранка—Николсона приводит к обратному перепаду у краёв. Этот эффект заметен уже при $r=2$, а при $r=10$ перепад несколько слоёв меняет знак. По одним глобальным минимуму и максимуму колебания в данном расчёте не видны.
3. **Безусловная устойчивость не гарантирует ни точности, ни отсутствия осцилляций.** Она ограничивает рост нормы, но не задаёт правильную скорость затухания коротких волн. Сравнение с малым шагом на той же сетке выявляет временную ошибку обеих неявных схем.